# Proyek Akhir Praktikum Big Data: TidarMart Data Platform
- **Nama:** Vandra Noprianus Sembiring
- **NPM:** 2505060006
- **SEED:** 60006 | **VARIAN:** 0 (Fokus Pembatalan dan Logistik)

## Tahap 1: Penyimpanan Terdistribusi dan Ingesti (HDFS)
**Tujuan:** Membangun struktur direktori *data lake* di HDFS (`/tidarmart/raw`, `/tidarmart/stream_input`, `/tidarmart/curated`, `/tidarmart/models`), mengunggah seluruh dataset hasil generator ke HDFS, serta memverifikasi ukuran berkas, jumlah blok, dan faktor replikasi.

In [5]:
# 1. Simpan metadata output generator untuk syarat pengumpulan
!echo "NPM 2505060006 | SEED 60006 | VARIAN pertanyaan bisnis = 0" > 2505060006_VandraNoprianusSembiring_Info.txt

# 2. Buat struktur direktori di HDFS
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi
!hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
!hdfs dfs -mkdir -p /tidarmart/raw/produk
!hdfs dfs -mkdir -p /tidarmart/stream_input
!hdfs dfs -mkdir -p /tidarmart/curated
!hdfs dfs -mkdir -p /tidarmart/models

# 3. Unggah dataset lokal dari data_proyek/ ke HDFS
!hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
!hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
!hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/
!hdfs dfs -put -f data_proyek/stream_input/*.csv /tidarmart/stream_input/

# 4. Tampilkan bukti ukuran berkas, jumlah blok, dan faktor replikasi (Karakteristik Volume)
!echo "=== UKURAN BERKAS DI HDFS ==="
!hdfs dfs -du -h /tidarmart/raw /tidarmart/stream_input
!echo "=== INFORMASI BLOK & REPLIKASI (FSCK) ==="
!hdfs fsck /tidarmart/raw -files -blocks

Starting resourcemanager
Starting nodemanagers
=== UKURAN BERKAS DI HDFS ===
6.0 M    6.0 M    /tidarmart/raw/pelanggan
684.2 K  684.2 K  /tidarmart/raw/produk
187.2 M  187.2 M  /tidarmart/raw/transaksi
479.5 K  479.5 K  /tidarmart/stream_input/stream_01.csv
479.4 K  479.4 K  /tidarmart/stream_input/stream_02.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_03.csv
479.4 K  479.4 K  /tidarmart/stream_input/stream_04.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_05.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_06.csv
478.5 K  478.5 K  /tidarmart/stream_input/stream_07.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_08.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_09.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_10.csv
479.0 K  479.0 K  /tidarmart/stream_input/stream_11.csv
479.8 K  479.8 K  /tidarmart/stream_input/stream_12.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_13.csv
478.9 K  478.9 K  /tidarmart/stream_input/stream_14.csv
479.5 K  479.

### Interpretasi Tahap 1
- **Pemisahan Direktori `raw` dan `curated`:** Direktori `/tidarmart/raw` menyimpan data mentah asli secara *immutable* (tidak diubah) sebagai *single source of truth* sehingga proses ETL dapat diulang atau diaudit kapan saja tanpa kehilangan data awal. Sementara `/tidarmart/curated` menyimpan data bersih berformat Parquet yang sudah tervalidasi agar siap digunakan untuk analitik dan *machine learning*.
- **Arti Replikasi = 1 pada VM Pseudo-Distributed:** Pada VM praktikum (1 node), faktor replikasi bernilai `1` karena hanya tersedia satu *DataNode* fisik sehingga tidak ada node lain untuk menampung salinan blok (tidak memiliki toleransi kegagalan perangkat keras). Pada klaster nyata multi-node, faktor replikasi umumnya diatur ke `3` agar data tetap tersedia meskipun salah satu node mengalami kerusakan.